# Урок 6. Многоклассовая классификация.

Посмотрим на примере алгоритма логистической регрессии и метода опорных векторов, как работать с различными методами многоклассовой классификации.

### 1.
Вспомните датасет Wine. Загрузите его, разделите на тренировочную и тестовую выборки (random_state=17), используя только [9, 11, 12] признаки.

In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

In [3]:
wine_dataset = load_wine()
wine_dataset['feature_names']

['alcohol',
 'malic_acid',
 'ash',
 'alcalinity_of_ash',
 'magnesium',
 'total_phenols',
 'flavanoids',
 'nonflavanoid_phenols',
 'proanthocyanins',
 'color_intensity',
 'hue',
 'od280/od315_of_diluted_wines',
 'proline']

In [4]:
x_train, x_test, y_train, y_test = train_test_split(wine_dataset.data[:, [9, 11, 12]], 
                                                    wine_dataset['target'],
                                                    random_state=17) 

print(f'X_train shape: {x_train.shape}, y_train shape: {y_train.shape},\n'
      f'X_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

X_train shape: (133, 3), y_train shape: (133,),
X_test shape: (45, 3), y_test shape: (45,)


**Задайте тип кросс-валидации с помощью StratifiedKFold: 5-кратная, random_state=17.**

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [8]:
#разбивка данных для кросс-валидации
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

### 2.
Обучите логистическую регрессию (LogisticRegression) с параметром C по умолчанию и random_state=17. Укажите гиперпараметр multi_class='ovr' - по умолчанию многие классификаторы используют именно его. С помощью cross_val_score сделайте кросс-валидацию (используйте объект skf) и выведите среднюю долю правильных ответов на ней (используйте функцию mean). Отдельно выведите долю правильных ответов на тестовой выборке.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
import numpy as np

In [44]:
#создаем LogisticRegression 
logreg = OneVsRestClassifier(LogisticRegression(random_state=17, max_iter=1000))

#кросс-валидация с помощью cross_val_score
cv_scores = cross_val_score(logreg, x_train, y_train, cv=skf, scoring='accuracy')

#сохраняем среднюю долю правильных ответов
mean_cv_accuracy = cv_scores.mean()

print(f"Средняя точность на кросс-валидации: {mean_cv_accuracy:.4f}")

Средняя точность на кросс-валидации: 0.9097


In [45]:
#обучаем модель на всей тренировочной выборке
logreg.fit(x_train, y_train)

#предсказания на тестовой выборке
test_predictions = logreg.predict(x_test)

#доля правильных ответов на тестовой выборке
test_accuracy = (test_predictions == y_test).mean()
print(f"Точность на тестовой выборке: {test_accuracy:.4f}")


Точность на тестовой выборке: 0.9111


### 3.
Обучите метод опорных векторов (SVC) с random_state=17 и остальными параметрами по умолчанию. Этот метод при мультиклассовой классификации также использует метод "ovr". Сделайте кросс-валидацию (используйте skf) и, как и в предыдущем пункте, выведите среднюю долю правильных ответов на ней. Отдельно выведите долю правильных ответов на тестовой выборке.

In [24]:
from sklearn.svm import SVC

In [46]:
#создаем SVC
svc_model = OneVsRestClassifier(SVC(random_state=17, max_iter=1000))

#кросс-валидация с помощью cross_val_score
cv = cross_val_score(svc_model, x_train, y_train, cv=skf, scoring='accuracy')

#сохраняем среднюю долю правильных ответов
mean_cv_accuracy_cv = cv.mean()

print(f"Средняя точность на кросс-валидации: {mean_cv_accuracy_cv:.4f}")


Средняя точность на кросс-валидации: 0.7148


In [47]:
#обучаем модель на всей тренировочной выборке
svc_model.fit(x_train, y_train)

#предсказания на тестовой выборке
test_predictions_svc = svc_model.predict(x_test)

#доля правильных ответов на тестовой выборке
test_accuracy_svc = (test_predictions_svc == y_test).mean()
print(f"Точность на тестовой выборке: {test_accuracy_svc:.4f}")


Точность на тестовой выборке: 0.6222


Как видно из полученной метрики, на тестовой выборке метод с гиперпараметрами по умолчанию работает явно намного хуже логистической регрессии. В целом, SVM достаточно плохо масштабируется на размер обучающего набора данных (как видно, даже с тремя признаками он работает не очень хорошо), но благодаря возможности выбора различных ядер (функций близости, которые помогают разделять данные) и другим гиперпараметрам SVM можно достаточно точно настроить под определенный вид данных. Подробнее на этом останавливаться в контексте данного урока не будем.

### 4.
Для предсказаний обеих моделей постройте матрицу ошибок (confusion matrix) и напишите, какие классы каждая из моделей путает больше всего между собой.

In [29]:
from sklearn.metrics import classification_report, confusion_matrix

In [52]:
#получаем матрицу ошибок
cm = confusion_matrix(y_test, test_predictions)
print("Матрица ошибок logreg :")
print(cm)
print(f"модель logreg путает класс 2 с классом 1")
#получаем матрицу ошибок
cm_1 = confusion_matrix(y_test, test_predictions_svc)
print("\nМатрица ошибок svc :")
print(cm_1)
print(f"модель svc вообще не определяет класс 2, всегда путая его с классами 0 и 1")

Матрица ошибок logreg :
[[ 9  0  0]
 [ 0 19  0]
 [ 0  4 13]]
модель logreg путает класс 2 с классом 1

Матрица ошибок svc :
[[ 9  0  0]
 [ 0 19  0]
 [ 2 15  0]]
модель svc вообще не определяет класс 2, всегда путая его с классами 0 и 1


### 5.
Для каждой модели выведите classification report.

In [55]:
#использование classification_report
report = classification_report(y_test, test_predictions)
print(f"classification report logreg: {report}")
report_svc = classification_report(y_test, test_predictions_svc)
print(f"\n\nclassification report svc: {report_svc}")

classification report logreg:               precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       0.83      1.00      0.90        19
           2       1.00      0.76      0.87        17

    accuracy                           0.91        45
   macro avg       0.94      0.92      0.92        45
weighted avg       0.93      0.91      0.91        45



classification report svc:               precision    recall  f1-score   support

           0       0.82      1.00      0.90         9
           1       0.56      1.00      0.72        19
           2       0.00      0.00      0.00        17

    accuracy                           0.62        45
   macro avg       0.46      0.67      0.54        45
weighted avg       0.40      0.62      0.48        45



C:\Users\user0907\AppData\Local\anaconda3\envs\my_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user0907\AppData\Local\anaconda3\envs\my_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user0907\AppData\Local\anaconda3\envs\my_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag